Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [1]:
NO_RECORDS = 100

In [2]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import get_doi, enrich_subjects_from_openaire, merge_subjects_from_new_method

d:\eosc_dc\metadata-warehouse\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [4]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")


if host == '':
    host = "127.0.0.1"
    user = os.getenv("POSTGRES_ADMIN")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [5]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
;
"""

In [6]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1773285923', 'title': 'IBS BAG', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/1773285923</identifier>\n    <datestamp>2024-09-09T06:00:11.703Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-1773285923</identifier><creators><creator><creatorName nameType="Personal">Christoph MUELLER DIECKMANN</creat

In [7]:
record_titles

['IBS BAG',
 'CHRONOS',
 'Mechanisms of maize biofortification induced by Fe nano fertilizers studied with EXAFS',
 'Sulf2',
 'Differential contrast tomography investigation of 3D microtexture for microstructure-driven high-cycle fatigue models through graph convolut',
 'In situ powder diffraction investigation of the adsorption selectivity of CO/H2 and N2/CO2 gas mixtures by single/mixed metal MOF-74-Ni,Mg',
 'Structural and chemical characterization of small alloy an ceramic reaction products recovered from high pressures and temperatures',
 'BAG Montpellier',
 'Characterizing Na ion battery',
 'Chromium speciation and distribution in vertical redox gradients of Ni laterite profiles',
 'Structural Biology at EMBL Grenoble, IBS-CIBB and IAB',
 '2D Shock focusing',
 'Operando investigation of cobalt oxide NPs reduction.',
 'The Netherlands BAG',
 '"Oxidation mechanism of copper in high voltage spinel materials for innovative Li-Ion battery"',
 'PROTEINS RELATED TO PATHOGENESIS OF DISEA

In [8]:
record_raw_metadata[2]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/1345032304</identifier>\n    <datestamp>2023-11-17T08:57:16.027Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-1345032304</identifier><creators><creator><creatorName nameType="Personal">Marta MARMIROLI</creatorName><givenName>Marta</givenName><familyName>Marmiroli</familyName><nameIdentifier nameIdentifierScheme="ORCID

In [9]:
get_doi(record_raw_metadata[2])

'10.15151/ESRF-ES-1345032304'

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [10]:
import logging
logging.getLogger("urllib3").setLevel(logging.WARNING)

In [11]:
doi = get_doi(record_raw_metadata[0]) 

if not doi:
    print("NO DOI")
else:
    print(enrich_subjects_from_openaire(doi))

['PaN', 'ID30B', 'MX-2586', 'x-ray probe', 'Particles, Nuclei and Fields', 'Macromolecular Crystallography']


In [12]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = get_doi(record_raw_metadata[i])
    if rec:
        ix_has_doi.append(i)

print(ix_has_doi)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]


In [13]:
id_to_enrich = [record_ids[i] for i in ix_has_doi]
doi_to_enrich = [get_doi(record_raw_metadata[i]) for i in ix_has_doi]

print(id_to_enrich)
print(doi_to_enrich)

['5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1773285923', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2113266150', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1345032304', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2031123823', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/891425266', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1028460808', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/962078919', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/96614964', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2297922202', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1308542889', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/837823686', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1030394047', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/632025980', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1006014569', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1347513065', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1460088562', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/569044652', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/93597902', 

In [14]:
# limit to smaller section
#id_to_enrich = id_to_enrich[:30]
#doi_to_enrich = doi_to_enrich[:30]

In [15]:
from utils.enrichment_methods import _openaire_batch_doi_resolve, _opeanaire_get_doi_subjects, connect_ids_with_subjects_by_doi, batch_enrich_subjects_from_openaire

api_result = _openaire_batch_doi_resolve(doi_to_enrich)

print(api_result[:3])

[{'authors': [{'fullName': 'GELIN, Muriel', 'name': 'Muriel', 'surname': 'GELIN', 'rank': 1, 'pid': None}, {'fullName': 'LEYRAT, Cedric', 'name': 'Cedric', 'surname': 'LEYRAT', 'rank': 2, 'pid': None}, {'fullName': 'LABESSE, Gilles', 'name': 'Gilles', 'surname': 'LABESSE', 'rank': 3, 'pid': None}, {'fullName': 'LIONNE, Corinne', 'name': 'Corinne', 'surname': 'LIONNE', 'rank': 4, 'pid': None}, {'fullName': 'LAMBEY, Paul', 'name': 'Paul', 'surname': 'LAMBEY', 'rank': 5, 'pid': None}, {'fullName': 'GRISON, Claire', 'name': 'Claire', 'surname': 'GRISON', 'rank': 6, 'pid': None}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'eng', 'label': 'English'}, 'countries': None, 'subjects': [{'subject': {'scheme': 'keyword', 'value': 'bm29'}, 'provenance': None}, {'subject': {'scheme': 'keyword', 'value': 'MX-1976'}, 'provenance': None}, {'subject': {'scheme': 'keyword', 'value': 'Macromolecular Crystallography'}, 'provenance': None}], 'mainTitle': 'BAG M

In [16]:
doi_subjects = _opeanaire_get_doi_subjects(api_result)
doi_subjects

[{'doi': '10.15151/esrf-es-96614964',
  'subjects': ['bm29', 'MX-1976', 'Macromolecular Crystallography']},
 {'doi': '10.15151/esrf-es-2031123823',
  'subjects': ['IH-MX-554', 'electron microscopy', 'CM02']},
 {'doi': '10.15151/esrf-es-2297922202', 'subjects': ['IH-MA-767', 'BM05']},
 {'doi': '10.15151/esrf-es-1345032304',
  'subjects': ['2. Zero hunger', 'BM08', 'A08-1-1108']},
 {'doi': '10.15151/esrf-es-2112610773',
  'subjects': ['PaN',
   'Earth Sciences',
   'x-ray probe',
   'Particles, Nuclei and Fields',
   'BM05',
   'BM18',
   'ES-1503']},
 {'doi': '10.15151/esrf-es-2113174295',
  'subjects': ['PaN',
   'Earth Sciences',
   'x-ray probe',
   'Particles, Nuclei and Fields',
   'BM05',
   'BM18',
   'ES-1503']},
 {'doi': '10.15151/esrf-es-2113174363',
  'subjects': ['PaN',
   'Earth Sciences',
   'x-ray probe',
   'Particles, Nuclei and Fields',
   'BM05',
   'BM18',
   'ES-1503']},
 {'doi': '10.15151/esrf-es-2112563831',
  'subjects': ['PaN',
   'Earth Sciences',
   'x-ray pro

In [17]:
id_dois = []
for ix in range(len(id_to_enrich)):
    id_dois.append(
        {
            'id': id_to_enrich[ix],
            'doi': doi_to_enrich[ix].lower(),
        }
    )
id_dois

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1773285923',
  'doi': '10.15151/esrf-es-1773285923'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2113266150',
  'doi': '10.15151/esrf-es-2113266150'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1345032304',
  'doi': '10.15151/esrf-es-1345032304'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2031123823',
  'doi': '10.15151/esrf-es-2031123823'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/891425266',
  'doi': '10.15151/esrf-es-891425266'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1028460808',
  'doi': '10.15151/esrf-es-1028460808'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/962078919',
  'doi': '10.15151/esrf-es-962078919'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/96614964',
  'doi': '10.15151/esrf-es-96614964'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2297922202',
  'doi': '10.15151/esrf-es-2297922202'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1308542889',

In [18]:
enriched_list = connect_ids_with_subjects_by_doi(id_dois, doi_subjects)
enriched_list

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1773285923',
  'doi': '10.15151/esrf-es-1773285923',
  'enriched_subjects': {'ID30B',
   'MX-2586',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2113266150',
  'doi': '10.15151/esrf-es-2113266150',
  'enriched_subjects': {'BM05',
   'BM18',
   'ES-1503',
   'Earth Sciences',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1345032304',
  'doi': '10.15151/esrf-es-1345032304',
  'enriched_subjects': {'2. Zero hunger', 'A08-1-1108', 'BM08'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2031123823',
  'doi': '10.15151/esrf-es-2031123823',
  'enriched_subjects': {'CM02', 'IH-MX-554', 'electron microscopy'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/891425266',
  'doi': '10.15151/esrf-es-891425266',
  'enriched_subjects': {'HC-4983',
   'Hard Condensed 

In [19]:
# Only succesfully enriched IDs and subjects.
# Could be useful for comparing how many ids we sent to be enriched vs
# how many actually got new subjects
[el for el in enriched_list if el['enriched_subjects'] is not None]

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1773285923',
  'doi': '10.15151/esrf-es-1773285923',
  'enriched_subjects': {'ID30B',
   'MX-2586',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2113266150',
  'doi': '10.15151/esrf-es-2113266150',
  'enriched_subjects': {'BM05',
   'BM18',
   'ES-1503',
   'Earth Sciences',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1345032304',
  'doi': '10.15151/esrf-es-1345032304',
  'enriched_subjects': {'2. Zero hunger', 'A08-1-1108', 'BM08'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2031123823',
  'doi': '10.15151/esrf-es-2031123823',
  'enriched_subjects': {'CM02', 'IH-MX-554', 'electron microscopy'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/891425266',
  'doi': '10.15151/esrf-es-891425266',
  'enriched_subjects': {'HC-4983',
   'Hard Condensed 

In [20]:
# All-in-one function

batch_enrich_subjects_from_openaire(id_dois)

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1773285923',
  'doi': '10.15151/esrf-es-1773285923',
  'enriched_subjects': {'ID30B',
   'MX-2586',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2113266150',
  'doi': '10.15151/esrf-es-2113266150',
  'enriched_subjects': {'BM05',
   'BM18',
   'ES-1503',
   'Earth Sciences',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1345032304',
  'doi': '10.15151/esrf-es-1345032304',
  'enriched_subjects': {'2. Zero hunger', 'A08-1-1108', 'BM08'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2031123823',
  'doi': '10.15151/esrf-es-2031123823',
  'enriched_subjects': {'CM02', 'IH-MX-554', 'electron microscopy'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/891425266',
  'doi': '10.15151/esrf-es-891425266',
  'enriched_subjects': {'HC-4983',
   'Hard Condensed 

In [21]:
# Comparison: Number of initial records vs number of records that have gained subjects via this method
enriched = batch_enrich_subjects_from_openaire(id_dois)
print(len([el['doi'] for el in enriched]))
print(len([el['doi'] for el in enriched if el['enriched_subjects'] is not None]))

100
100


In [22]:
# Taking a look at the metadata initially obtained from DB, subset of few records
full_info_enriched = [el for el in enriched if el['enriched_subjects'] is not None]

first_enriched = full_info_enriched[:10]

In [23]:
# indexes that have been enriched
ix_original = [record_ids.index(rec['id']) for rec in first_enriched]
ix_original

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [24]:
first_enriched

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1773285923',
  'doi': '10.15151/esrf-es-1773285923',
  'enriched_subjects': {'ID30B',
   'MX-2586',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2113266150',
  'doi': '10.15151/esrf-es-2113266150',
  'enriched_subjects': {'BM05',
   'BM18',
   'ES-1503',
   'Earth Sciences',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1345032304',
  'doi': '10.15151/esrf-es-1345032304',
  'enriched_subjects': {'2. Zero hunger', 'A08-1-1108', 'BM08'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2031123823',
  'doi': '10.15151/esrf-es-2031123823',
  'enriched_subjects': {'CM02', 'IH-MX-554', 'electron microscopy'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/891425266',
  'doi': '10.15151/esrf-es-891425266',
  'enriched_subjects': {'HC-4983',
   'Hard Condensed 

In [25]:
[record_raw_metadata[i] for i in ix_original]

['<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/1773285923</identifier>\n    <datestamp>2024-09-09T06:00:11.703Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-1773285923</identifier><creators><creator><creatorName nameType="Personal">Christoph MUELLER DIECKMANN</creatorName><givenName>Christoph</givenName><familyName>Mueller-Dieckmann</familyName></creator><creator

## OpenAlex API
https://developers.openalex.org/
https://developers.openalex.org/guides/authentication

Get multiple works by DOI

use | between DOIs, up to 100 DOIs at once!!
check out how the results look then.

https://api.openalex.org/works?filter=doi:https://doi.org/10.1234/a|https://doi.org/10.1234/b|https://doi.org/10.1234/c&per_page=100



Use the select parameter to return only specific fields:
https://api.openalex.org/works?select=id,doi,display_name

In [26]:
OpenAlex_API_key = os.getenv("OPENALEX_API_KEY")

In [27]:
OPENALEX_API_URL = "https://api.openalex.org/works/"

doi = '10.15151/esrf-es-1473312981'

# We have to append the ID directly in the path
query_url = OPENALEX_API_URL + "https://doi.org/" + doi

response = requests.get(
    query_url,
    params={"api_key": OpenAlex_API_key}, 
    timeout=30
)

data = response.json()


In [28]:
oa_DOIS = [
    '10.15151/esrf-es-1001518064',
    '10.15151/esrf-es-2323561777',
    '10.15151/esrf-es-2323561778',
    '10.15151/esrf-es-2323561877',
    'https://doi.org/10.5286/ISIS.E.100051344',
    '10.34894/AUUSS2',
    '10.17026/DANS-28H-C53V',
    '10.34894/VLSN1K',
    '10.15151/ESRF-ES-1030397492',
    '10.17026/AR/DMHU53',
    '10.17026/AR/GHYQTO',
    '10.17026/DANS-ZU2-HF3N',
    '10.17026/DANS-ZTM-7SA5',
    '10.17026/DANS-ZUK-HN2Z',
    'https://doi.org/10.1016/j.paid.2021.110888',
    '10.23662/fors-ds-686-1',
    '10.1056/NEJMoa2034577',
    '10.1056/NEJMoa2021436',
    '10.1056/NEJMoa1511939',
    '10.1056/NEJMoa1800389'
]

the API result object can have a result for a given DOI, but that result doesn't contain any keywords = doesn't contribute to enrichment
Hence the len(api_result) can be larger than len(enriched_dois)

Additionaly if we set a high keyword score threshold, a certain result MIGHT have keywords but none are relevant enough - that will also not contribute to enrichment

In [29]:
from utils.enrichment_methods import batch_enrich_subjects_from_openalex

enriched_results = batch_enrich_subjects_from_openalex(
    id_dois,
    OpenAlex_API_key,
    score_treshold = 0.6
)

enriched_results = [el for el in enriched_results if el['enriched_subjects']]
print(len(enriched_results))
enriched_results

9


[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2334407190',
  'doi': '10.15151/esrf-es-2334407190',
  'enriched_subjects': {'Multiferroics', 'Strain (injury)'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2312891623',
  'doi': '10.15151/esrf-es-2312891623',
  'enriched_subjects': {'Cathode'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2331197866',
  'doi': '10.15151/esrf-es-2331197866',
  'enriched_subjects': {'Absorption spectroscopy',
   'Nickel',
   'X-ray absorption spectroscopy'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2383302089',
  'doi': '10.15151/esrf-es-2383302089',
  'enriched_subjects': {'Colloid'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2312932551',
  'doi': '10.15151/esrf-es-2312932551',
  'enriched_subjects': {'Absorption (acoustics)',
   'Absorption spectroscopy',
   'X-ray absorption spectroscopy'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2354226726',
  'doi': '10.15151/esrf-es-2354226726',
  'enriched_subjects

In [30]:
# Showcase  - manually input DOIs

test_dois = [
    {'id': "testid1",
     'doi': '10.1016/j.paid.2021.110888'},
    {'id': "testid2",
     'doi': '10.23662/fors-ds-686-1'}
]

enriched_results = batch_enrich_subjects_from_openalex(
    test_dois,
    OpenAlex_API_key,
    score_treshold = 0.7
)

enriched_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'Conscientiousness',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': set()}]

In [31]:
# Compare openalex with openaire:
oaire_results = batch_enrich_subjects_from_openaire(test_dois)
oaire_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}}]

In [32]:
merge_subjects_from_new_method(enriched_results, oaire_results)

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality',
   'Conscientiousness',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}}]

# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

In [33]:
## DATACITE REST API

DATACITE_API_URL = "https://api.datacite.org/dois"

DataCite provides an additional "enriched" option for fetching doi-by-doi, could be interesting.

In [34]:
def _datacite_batch_doi_resolve(dois: list[str]) -> list[dict]:
    all_results = []

    clean_dois = [
        doi.replace("https://doi.org/", "")
           .replace("http://doi.org/", "")
        for doi in dois
    ]

    for doi in clean_dois:
        response = requests.get(
            f"{DATACITE_API_URL}/{doi}",
            params={"disable-facets": "true"},
            timeout=30,
        )
        if response.status_code == 404:
            continue
        response.raise_for_status()
        all_results.append(response.json().get("data", {}))

    return all_results

In [35]:
DC_res = _datacite_batch_doi_resolve(doi_to_enrich)

In [36]:
DC_res

[{'id': '10.15151/esrf-es-1773285923',
  'type': 'dois',
  'attributes': {'doi': '10.15151/esrf-es-1773285923',
   'prefix': '10.15151',
   'suffix': 'esrf-es-1773285923',
   'identifiers': [],
   'alternateIdentifiers': [],
   'creators': [{'name': 'De La Mora, Eugenio',
     'nameType': 'Personal',
     'givenName': 'Eugenio',
     'familyName': 'De La Mora',
     'affiliation': ['Institut de Biologie Structurale - IBS,Metalloproteins Unit,71 avenue des Martyrs,CS 10090,38044 GRENOBLE Cedex 9,38044,GRENOBLE,FRANCE'],
     'nameIdentifiers': [{'schemeUri': 'https://orcid.org',
       'nameIdentifier': 'https://orcid.org/0000-0001-6945-7357',
       'nameIdentifierScheme': 'ORCID'}]},
    {'name': 'Ruel, Jérémie',
     'nameType': 'Personal',
     'givenName': 'Jérémie',
     'familyName': 'Ruel',
     'affiliation': ['Institut de Biologie Structurale - IBS,Metalloproteins Unit,71 avenue des Martyrs,CS 10090,38044 GRENOBLE Cedex 9,38044,GRENOBLE,FRANCE'],
     'nameIdentifiers': [{'sch

In [37]:
[el['attributes']['subjects'] for el in DC_res]

[[{'subject': 'Macromolecular Crystallography',
   'subjectScheme': 'Proposal Type Description'},
  {'subject': 'MX-2586', 'subjectScheme': 'Proposal'},
  {'subject': 'ID30B', 'subjectScheme': 'Instrument'},
  {'subject': 'x-ray probe',
   'valueUri': 'http://purl.org/pan-science/PaNET/PaNET01012',
   'schemeUri': 'http://purl.org/pan-science/PaNET/PaNET.owl',
   'subjectScheme': 'PaNET',
   'classificationCode': 'PaNET01012'}],
 [{'subject': 'Earth Sciences', 'subjectScheme': 'Proposal Type Description'},
  {'subject': 'ES-1503', 'subjectScheme': 'Proposal'},
  {'subject': 'BM18', 'subjectScheme': 'Instrument'},
  {'subject': 'x-ray probe',
   'valueUri': 'http://purl.org/pan-science/PaNET/PaNET01012',
   'schemeUri': 'http://purl.org/pan-science/PaNET/PaNET.owl',
   'subjectScheme': 'PaNET',
   'classificationCode': 'PaNET01012'}],
 [{'subjectScheme': 'Proposal Type Description'},
  {'subject': 'A08-1-1108', 'subjectScheme': 'Proposal'},
  {'subject': 'BM08', 'subjectScheme': 'Instru

In [38]:
slist = [el['attributes']['subjects'] for el in DC_res]

[[el.get('subject') for el in sublist] for sublist in slist]

[['Macromolecular Crystallography', 'MX-2586', 'ID30B', 'x-ray probe'],
 ['Earth Sciences', 'ES-1503', 'BM18', 'x-ray probe'],
 [None, 'A08-1-1108', 'BM08'],
 [None, 'IH-MX-554', 'CM02', 'electron microscopy'],
 ['Hard Condensed Matter Science', 'HC-4983', 'ID11'],
 [None, 'A01-2-1283', 'BM01'],
 ['Chemistry', 'CH-6365', 'ID11'],
 ['Macromolecular Crystallography', 'MX-1976', 'bm29'],
 [None, 'IH-MA-767', 'BM05'],
 ['Environment', 'EV-495', 'BM20'],
 ['Macromolecular Crystallography', 'MX-2357', 'ID30A-1'],
 ['Hard Condensed Matter Science', 'HC-5095', 'ID09'],
 ['In-house Chemistry', 'IH-CH-1578', 'ID10'],
 ['Macromolecular Crystallography', 'MX-2407', 'ID23-1'],
 ['Applied Material Science', 'MA-5900', 'BM23'],
 ['Macromolecular Crystallography', 'MX-2591', 'ID23-2'],
 ['Chemistry', 'CH-6158', 'ID13'],
 ['Macromolecular Crystallography', 'MX-1979', 'id23-1'],
 ['Soft Condensed Matter Science', 'SC-5189', 'ID13'],
 ['Macromolecular Crystallography', 'MX-2521', 'BM07'],
 ['Chemistry', 

In [39]:
def _datacite_get_doi_subjects(response: list[dict]):
    """
    Transform API response into a list of DOIs and their subject-sets.

    OpenAlex responses have 3 machine-derived fields of interest:
    topics, keywords, concepts

    Concepts are a legacy version of keywords so we'll skip those.
    We'll consider keywords and concepts as subjects of interest. We'll further filter them out by similarity score.
    
    :param response: List[dict], the API response
    :param score_treshold: float, minimum similarity score a topic/keyword must have to be included as a subject
    :return: List[dict]
        dict:
            DOI - DOI that is linked to at least one subject
            [Subjects] - all subjects for the given DOI
    """
    if response == []:
        # no DOIs with any results
        return []
    
    # sets of DOIs with relevant sets of subjects. Later, spread to individual DOIs
    enriched_dois = []
    enriched_subjects = []

    for el in response:
        sub_list = el['attributes']['subjects']

        if (sub_list == []):
            # this element doesn't contribute to enriching subjects
            continue

        # It's possible that some dicts do not have the subject key
        subjects = [item.get('subject') for item in sub_list if item.get('subject')]

        if not subjects:
            continue
        
        current_doi = el['id']

        # Strip DOI from prefixes and make it lowercase
        current_doi = current_doi.replace("https://doi.org/", "").replace("http://doi.org/", "").lower()

        enriched_dois.append(current_doi)

        enriched_subjects.append(subjects)

    # Create a list of dicts as output

    doi_subjects = []

    for ix in range(len(enriched_dois)):
        doi_sub_pair = {'doi': enriched_dois[ix],
                        'subjects': enriched_subjects[ix]}
        doi_subjects.append(doi_sub_pair)

    return doi_subjects

In [40]:
_datacite_get_doi_subjects(DC_res)

[{'doi': '10.15151/esrf-es-1773285923',
  'subjects': ['Macromolecular Crystallography',
   'MX-2586',
   'ID30B',
   'x-ray probe']},
 {'doi': '10.15151/esrf-es-2113266150',
  'subjects': ['Earth Sciences', 'ES-1503', 'BM18', 'x-ray probe']},
 {'doi': '10.15151/esrf-es-1345032304', 'subjects': ['A08-1-1108', 'BM08']},
 {'doi': '10.15151/esrf-es-2031123823',
  'subjects': ['IH-MX-554', 'CM02', 'electron microscopy']},
 {'doi': '10.15151/esrf-es-891425266',
  'subjects': ['Hard Condensed Matter Science', 'HC-4983', 'ID11']},
 {'doi': '10.15151/esrf-es-1028460808', 'subjects': ['A01-2-1283', 'BM01']},
 {'doi': '10.15151/esrf-es-962078919',
  'subjects': ['Chemistry', 'CH-6365', 'ID11']},
 {'doi': '10.15151/esrf-es-96614964',
  'subjects': ['Macromolecular Crystallography', 'MX-1976', 'bm29']},
 {'doi': '10.15151/esrf-es-2297922202', 'subjects': ['IH-MA-767', 'BM05']},
 {'doi': '10.15151/esrf-es-1308542889',
  'subjects': ['Environment', 'EV-495', 'BM20']},
 {'doi': '10.15151/esrf-es-8378

In [41]:
import copy

def batch_enrich_subjects_from_datacite(records_id_doi: list[dict]):
    """
    :param records_id_doi: List[dict: 
        {'id': str, record id from database,
        'doi': str, record DOI
        }
    ]
    :return: List[dict] like input, with new key:
            'enriched_subjects': None/set[str]
    """
    
    dois_to_enrich = [rec['doi'] for rec in records_id_doi]
    api_result = _datacite_batch_doi_resolve(dois_to_enrich)

    doi_subjects = _datacite_get_doi_subjects(api_result)

    # Don't alter the input
    output_list = copy.deepcopy(records_id_doi)
    output_list = connect_ids_with_subjects_by_doi(output_list, doi_subjects)

    # List of following:
    #[
    # id: str
    # doi: str
    # enriched_subjects: {str}
    #]
    return output_list

In [42]:
DC_enriched = batch_enrich_subjects_from_datacite(id_dois)

In [43]:
[el for el in DC_enriched if el['enriched_subjects']]

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1773285923',
  'doi': '10.15151/esrf-es-1773285923',
  'enriched_subjects': {'ID30B',
   'MX-2586',
   'Macromolecular Crystallography',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2113266150',
  'doi': '10.15151/esrf-es-2113266150',
  'enriched_subjects': {'BM18', 'ES-1503', 'Earth Sciences', 'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1345032304',
  'doi': '10.15151/esrf-es-1345032304',
  'enriched_subjects': {'A08-1-1108', 'BM08'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2031123823',
  'doi': '10.15151/esrf-es-2031123823',
  'enriched_subjects': {'CM02', 'IH-MX-554', 'electron microscopy'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/891425266',
  'doi': '10.15151/esrf-es-891425266',
  'enriched_subjects': {'HC-4983', 'Hard Condensed Matter Science', 'ID11'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1028460808',
  'doi': '10.15151/esrf-es-1028460808',


In [44]:
len([el for el in DC_enriched if el['enriched_subjects']])

100

In [45]:
# All in one method:
from utils.enrichment_methods import batch_enrich_subjects_from_datacite
batch_enrich_subjects_from_datacite(test_dois)

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': set()},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': set()}]

# Related DOIs  
Extract related identifiers that are also DOIs. If they are in the database, use their subjects. If not, use DOI-resolving methods on them.

Note:
a DOI must start with 10. prefix. I've seen some related DOIs that start with 0.  , if I add a 1 to the start it resolves properly.

## SQL query to see records with related DOIs   
```
--- get records with related identifiers of any tipe
WITH query AS (
    SELECT
        raw_metadata,
		doi as record_doi,
        xpath($$(//*[local-name()='relatedIdentifier'][@relatedIdentifierType='DOI'])/text()$$, raw_metadata)::text[] AS related
    FROM public.records
)
SELECT *
FROM query
WHERE array_length(related, 1) > 0
LIMIT 1000;
```

47k records have SOME relatedIdentifiers - lots of them ISSN or proprietary ids

In [46]:
# Let's try out some records with related DOIs, for testing we won't limit ourselves to records with no raw_subjects.

query_related_dois = """
WITH query AS (
    SELECT
        id,
        raw_metadata,
		doi as record_doi,
        ARRAY(
            SELECT jsonb_array_elements(datacite_json->'subjects')->>'subject'
        ) AS subjects,
        xpath($$(//*[local-name()='relatedIdentifier'][@relatedIdentifierType='DOI'])/text()$$, raw_metadata)::text[] AS related
    FROM public.records
)
SELECT *
FROM query
WHERE array_length(related, 1) > 0
LIMIT 100;
"""

In [47]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_related_dois)
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_subjects = [row["subjects"] for row in results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]
        record_related_dois = [row["related"] for row in results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '557d465d-8e79-45ed-9e44-60e105fb0ecb::kbco_406', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:dabar.srce.hr:kbco_406</identifier>\n    <datestamp>2024-12-09</datestamp>\n  </header>\n  <metadata>\n    <datacite:resource xmlns:datacite="http://datacite.org/schema/kernel-4" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:dabar="http://dabar.srce.hr/standards/schema/1.0" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.4/metadata.xsd">\n      <datacite:identifier identifierType="DOI">10.1016/j.cca.2024.119758</datacite:identifier>\n      <datacite:alternateIdentifiers>\n        <datacite:alternateIdentifier alternateIdentifierType="URL">https://urn.nsk.hr/urn:nbn:hr:239:263661</datacite:alternateIdentifier>\n        <datacite:alternateIdentifier alternateIdentifierType="LOCAL">kbco:406</datacite:alternateIdent

In [48]:
results

[{'id': '557d465d-8e79-45ed-9e44-60e105fb0ecb::kbco_406',
  'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:dabar.srce.hr:kbco_406</identifier>\n    <datestamp>2024-12-09</datestamp>\n  </header>\n  <metadata>\n    <datacite:resource xmlns:datacite="http://datacite.org/schema/kernel-4" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:dabar="http://dabar.srce.hr/standards/schema/1.0" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.4/metadata.xsd">\n      <datacite:identifier identifierType="DOI">10.1016/j.cca.2024.119758</datacite:identifier>\n      <datacite:alternateIdentifiers>\n        <datacite:alternateIdentifier alternateIdentifierType="URL">https://urn.nsk.hr/urn:nbn:hr:239:263661</datacite:alternateIdentifier>\n        <datacite:alternateIdentifier alternateIdentifierType="LOCAL">kbco:406</datacite:alternateIde

In [49]:
# For a specific DOI set, search our datasetdb and retrieve subjects if any.
# Maybe a record references/is part of another dataset so the subjects are relevant too.

# This query works for a list of DOIs -  if a record has multiples

query_subjects_by_doi = """
WITH query AS (
    SELECT
    id,
        ARRAY(
            SELECT jsonb_array_elements(datacite_json->'subjects')->>'subject'
        ) AS subjects
    FROM public.records
)
SELECT subjects
FROM query
WHERE query.id = ANY(%s)
LIMIT 100;
"""

In [50]:
first_related_doi = record_related_dois[0]
first_related_doi

['10.1016/j.cca.2024.119758']

In [51]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_subjects_by_doi, (first_related_doi,))
        
        record_res = cur.fetchall()
except:
    print(f"Query error: {e}")
        

record_res

[]

In [52]:
# Go through all test cases

related_subjects = []

for i in range(len(record_ids)):
    try:
        with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
            cur = conn.cursor()
            related_dois = record_related_dois[i]
            cur.execute(query_subjects_by_doi, (related_dois,))
            
            record_res = cur.fetchall()
            related_subjects.append(record_res)
    except:
        print(f"Query error: {e}")
        

In [53]:
# No hits
[res for res in related_subjects if res]

[]

In [54]:
# Extra: go through our DB
# and ALSO retrieve all subjects from openaire and openalex for those individual DOIs

related_subjects = []

for i in range(len(record_ids)):
    try:
        with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
            cur = conn.cursor()
            related_dois = record_related_dois[i]
            cur.execute(query_subjects_by_doi, (related_dois,))
            
            record_res = cur.fetchall()
            record_res = set(record_res)
            # We need an id-doi pairing for each doi, we'll make fake ids since we won't use them for joining later
            id_dois = [{'id': str(i), 'doi': doi,} for i, doi in enumerate(related_dois)]
            print(id_dois)

            openaire = batch_enrich_subjects_from_openaire(id_dois)
            openaire_subjects = set().union(*(el["enriched_subjects"] for el in openaire))
            
            openalex = batch_enrich_subjects_from_openalex(id_dois, OpenAlex_API_key)
            openalex_subjects = set().union(*(el["enriched_subjects"] for el in openalex))

            # Union of all three
            record_res = record_res | openaire_subjects | openalex_subjects

            related_subjects.append(record_res)
    except:
        print(f"Query error: {e}")
        

[{'id': '0', 'doi': '10.1016/j.cca.2024.119758'}]
[{'id': '0', 'doi': '10.3390/ph17101259'}]
[{'id': '0', 'doi': '10.3390/pharmaceutics17101299'}]
[{'id': '0', 'doi': '10.1039/d5ra05470b'}]
[{'id': '0', 'doi': '10.1002/cmtd.202400025'}]
[{'id': '0', 'doi': '10.3390/molecules30183682'}]
[{'id': '0', 'doi': '10.1039/D5QM00456J'}]
[{'id': '0', 'doi': '10.23919/ECC54610.2021.9654836'}]
[{'id': '0', 'doi': '10.1051/0004-6361/202450082'}]
[{'id': '0', 'doi': '10.1186/s12874-020-00990-4'}]
[{'id': '0', 'doi': '10.1103/PhysRevLett.134.191002'}]
[{'id': '0', 'doi': '10.1088/1475-7516/2024/11/012'}]
[{'id': '0', 'doi': '10.1039/d5dt02399h'}]
[{'id': '0', 'doi': '10.48550/arXiv.2505.16880'}]
[{'id': '0', 'doi': '10.1038/s41598-021-90690-w'}]
[{'id': '0', 'doi': '10.1021/jasms.1c00200'}]
[{'id': '0', 'doi': '10.1016/j.jpba.2025.116664'}]
[{'id': '0', 'doi': '10.3390/pharmaceutics17101299'}]
[{'id': '0', 'doi': '10.32762/zr.27.1.9'}]
[{'id': '0', 'doi': '10.1039/D5CC04696C'}]
[{'id': '0', 'doi': '1

In [55]:
related_subjects

[{'Analytic sample preparation methods',
  'Antigens, CD19',
  'B-cell',
  'CD19 antigen',
  'Chemistry',
  'Chronic lymphocytic leukemia',
  'Humans',
  'Leukemia Lymphocytic Chronic',
  'Leukemia, Lymphocytic, Chronic, B-Cell',
  'Lymphocyte',
  'Lymphocytes',
  'MALDI-MS',
  'Mass spectrometry',
  'Metabolomics',
  'Single-Cell Analysis',
  'Single-cell analysis',
  'Spectrometry, Mass, Matrix-Assisted Laser Desorption-Ionization'},
 {'3',
  '3,5-disubstituted hydantoins',
  '3,5-disubstituted hydantoins; preparative HPLC separation; NMR analysis; absolute configuration; CD/VCD; antiproliferative activity; in silico biological profiling',
  '5-disubstituted hydantoins',
  'Absolute configuration',
  'Article',
  'CD/VCD',
  'Medicine',
  'NMR analysis',
  'Pharmacy and materia medica',
  'R',
  'RS1-441',
  'Synthesis and Characterization of Heterocyclic Compounds',
  'Synthesis and biological activity',
  'Synthesis of heterocyclic compounds',
  'absolute configuration',
  'antipro